In [1]:
import nilearn as nl
import numpy as np
import matplotlib
import pandas as pd
import os
import nibabel as nib
import matplotlib.pyplot as plt
import openpyxl as xl

from nilearn.glm.first_level import make_first_level_design_matrix
from nilearn.plotting import plot_design_matrix, plot_contrast_matrix
from nilearn.glm.first_level import FirstLevelModel
from nilearn.masking import compute_epi_mask, apply_mask, unmask
from nilearn.image import index_img, resample_to_img, math_img, mean_img, get_data
from nilearn.plotting import plot_stat_map, show
from nilearn import image, masking
from nilearn import plotting
from nilearn.datasets import load_mni152_brain_mask
from pandas import read_excel

from nilearn.glm.second_level import SecondLevelModel
from scipy.stats import norm

from nilearn.glm import threshold_stats_img

In [66]:
textValueFileDF = pd.read_excel("/Users/silvycollin/Documents/GitHub/wholeBrain_narratives_MAC_MFT/text/foundationScores/lucy_MFT_MAC.xlsx")

sentenceValues = textValueFileDF[['MAC_a_fairness_virtue',
                               'MAC_a_group_virtue',
                               'MAC_a_deference_virtue',
                               'MAC_a_heroism_virtue',
                               'MAC_a_reciprocity_virtue',
                               'MAC_a_family_virtue',
                               'MAC_a_property_virtue']]



In [67]:
sentenceValues

,MAC_a_fairness_virtue,MAC_a_group_virtue,MAC_a_deference_virtue,MAC_a_heroism_virtue,MAC_a_reciprocity_virtue,MAC_a_family_virtue,MAC_a_property_virtue
0,0.079832,0.037975,0.079562,0.043948,0.011364,0.072746,0.015190
1,0.000000,0.000000,0.000000,0.000000,0.194631,0.109589,0.105960
2,0.083224,0.092021,0.062016,0.030651,0.067511,0.037476,0.130252
3,0.033582,0.000000,0.024096,0.029412,0.077075,0.202103,0.000000
4,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
...,...,...,...,...,...,...,...
95,0.000000,0.000000,0.000000,0.000000,0.199020,0.093496,0.000000
96,0.073037,0.103800,0.056497,0.040469,0.052980,0.093011,0.000000
97,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
98,0.141838,0.093088,0.087316,0.067432,0.075422,0.065321,0.047263


In [68]:

events = pd.DataFrame( {"trial_type":range(0,len(sentenceValues))})

# dtermine the max column for each row
max_column = sentenceValues.idxmax(axis=1)

# Determine rows where all 7 columns are 0.00 (to two decimal places)
all_zero_mask = (sentenceValues.round(2) == 0).all(axis=1)
# Replace column name with "baseline" for those rows
max_column[all_zero_mask] = "baseline"

max_columns_list = max_column.tolist()

events['trial_type'] = max_columns_list

In [69]:
events

,trial_type
0,MAC_a_fairness_virtue
1,MAC_a_reciprocity_virtue
2,MAC_a_property_virtue
3,MAC_a_family_virtue
4,baseline
...,...
95,MAC_a_reciprocity_virtue
96,MAC_a_group_virtue
97,baseline
98,MAC_a_fairness_virtue


In [70]:
baseline_count = (events['trial_type'] == 'baseline').sum()
print('number of baseline: ',baseline_count)

fai_count = (events['trial_type'] == 'MAC_a_fairness_virtue').sum()
print('number of fairness virtue: ',fai_count)

fam = (events['trial_type'] == 'MAC_a_family_virtue').sum()
print('number of family virtue: ',fam)

her_count = (events['trial_type'] == 'MAC_a_heroism_virtue').sum()
print('number of heroism virtue: ',her_count)

rec_count = (events['trial_type'] == 'MAC_a_reciprocity_virtue').sum()
print('number of reciprocity virtue: ',rec_count)

gr_count = (events['trial_type'] == 'MAC_a_group_virtue').sum()
print('number of group virtue: ',gr_count)

prop_count = (events['trial_type'] == 'MAC_a_property_virtue').sum()
print('number of property virtue: ',prop_count)

def_count = (events['trial_type'] == 'MAC_a_deference_virtue').sum()
print('number of deference virtue: ',def_count)

number of baseline:  26
number of fairness virtue:  6
number of family virtue:  19
number of heroism virtue:  5
number of reciprocity virtue:  15
number of group virtue:  18
number of property virtue:  3
number of deference virtue:  8


In [71]:
sentenceValues_vice = textValueFileDF[['MAC_a_fairness_vice',
                               'MAC_a_group_vice',
                               'MAC_a_deference_vice',
                               'MAC_a_heroism_vice',
                               'MAC_a_reciprocity_vice',
                               'MAC_a_family_vice',
                               'MAC_a_property_vice']]


events = pd.DataFrame( {"trial_type":range(0,len(sentenceValues_vice))})

# dtermine the max column for each row
max_column = sentenceValues_vice.idxmax(axis=1)

# Determine rows where all 7 columns are 0.00 (to two decimal places)
all_zero_mask = (sentenceValues_vice.round(2) == 0).all(axis=1)
# Replace column name with "baseline" for those rows
max_column[all_zero_mask] = "baseline"

max_columns_list = max_column.tolist()

events['trial_type'] = max_columns_list

In [72]:
baseline_count = (events['trial_type'] == 'baseline').sum()
print('number of baseline: ',baseline_count)

fai_count = (events['trial_type'] == 'MAC_a_fairness_vice').sum()
print('number of fairness vice: ',fai_count)

fam = (events['trial_type'] == 'MAC_a_family_vice').sum()
print('number of family vice: ',fam)

her_count = (events['trial_type'] == 'MAC_a_heroism_vice').sum()
print('number of heroism vice: ',her_count)

rec_count = (events['trial_type'] == 'MAC_a_reciprocity_vice').sum()
print('number of reciprocity vice: ',rec_count)

gr_count = (events['trial_type'] == 'MAC_a_group_vice').sum()
print('number of group vice: ',gr_count)

prop_count = (events['trial_type'] == 'MAC_a_property_vice').sum()
print('number of property vice: ',prop_count)

def_count = (events['trial_type'] == 'MAC_a_deference_vice').sum()
print('number of deference vice: ',def_count)

number of baseline:  26
number of fairness vice:  20
number of family vice:  14
number of heroism vice:  11
number of reciprocity vice:  7
number of group vice:  9
number of property vice:  10
number of deference vice:  3


In [73]:
sentenceValues_virtueVice = textValueFileDF[['MAC_a_fairness_virtue',
                               'MAC_a_group_virtue',
                               'MAC_a_deference_virtue',
                               'MAC_a_heroism_virtue',
                               'MAC_a_reciprocity_virtue',
                               'MAC_a_family_virtue',
                               'MAC_a_property_virtue',
                                 'MAC_a_fairness_vice',
                               'MAC_a_group_vice',
                               'MAC_a_deference_vice',
                               'MAC_a_heroism_vice',
                               'MAC_a_reciprocity_vice',
                               'MAC_a_family_vice',
                               'MAC_a_property_vice']]

events = pd.DataFrame( {"trial_type":range(0,len(sentenceValues_virtueVice))})

# dtermine the max column for each row
max_column = sentenceValues_virtueVice.idxmax(axis=1)

# Determine rows where all 7 columns are 0.00 (to two decimal places)
all_zero_mask = (sentenceValues_virtueVice.round(2) == 0).all(axis=1)
# Replace column name with "baseline" for those rows
max_column[all_zero_mask] = "baseline"

max_columns_list = max_column.tolist()

events['trial_type'] = max_columns_list


# counting baseline all 14 foundations (across vice and virtue)
baseline_count = (events['trial_type'] == 'baseline').sum()
print('number of baseline: ',baseline_count)

number of baseline:  24
